# Lab 2: Spectral Domain Image Transforms

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Learning Objectives
* **Dimensionality Reduction:** Mathematically compress a 13-band Sentinel-2 image using Principal Component Analysis (PCA).
* **Statistical Exploration:** Analyze the covariance and correlation of satellite bands.
* **Physical Interpretability:** Map abstract Principal Components to physical handcrafted features (Albedo, NDVI, NDWI) using quantitative L1 loss metrics.
* **Variance Interpretation:** Analyze what PCA isolates in the lower-variance components.

### Autograding Advisory
This notebook is designed for automated grading. You will find `TODO` comments throughout the code cells; ensure you assign your mathematical results to the exact variable names requested. 

At the end of the notebook, there is a submission dictionary. **You are expected to modify this dictionary structure** to include your student IDs, your answers to the analysis questions, and any notes you wish to leave for the instructor.


In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

### 1. Data Loading & Flattening
We are loading `hazy_cufe.tiff`, a 13-band Level-1C Sentinel-2 image. To perform linear algebra, we must flatten the spatial dimensions into a 2D matrix.

In [ ]:
image_path = "hazy_cufe.tiff"

with rasterio.open(image_path) as src:
    img_3d = src.read().astype(np.float32)

print(f"Original Shape: {img_3d.shape}")

# TODO: Flatten img_3d into a 2D matrix named 'X'. 
# Think carefully about how dimensions align in memory during a reshape operation.

# print(f"Flattened Shape: {X.shape}") 


### 2. Statistical Exploration
Multispectral bands are often highly correlated. Let's prove it statistically.

In [ ]:
# TODO: Calculate the 13x13 correlation matrix.
# TODO: Find the maximum and minimum correlation values between any two *different* bands.
# Store them in variables named 'max_corr' and 'min_corr'.

max_corr = 0.0 
min_corr = 0.0 

print(f"Max Correlation: {max_corr:.2f}")
print(f"Min Correlation: {min_corr:.2f}")


### 3. Principal Component Analysis (PCA)
Let's compress the data using PCA.

In [ ]:
# TODO: Mean-center the data, apply PCA, and extract the principal components.
# TODO: Calculate the percentage of variance explained by the first three components.
# Store these percentages in 'var_pc1', 'var_pc2', and 'var_pc3' (e.g., 90.5 for 90.5%).
# TODO: Plot the Cumulative Explained Variance.

var_pc1 = 0.0 
var_pc2 = 0.0 
var_pc3 = 0.0


### 4. Physical Interpretability
Let's calculate three physical metrics from our flattened original data `X`:
* `Brightness` = Mean of Visible + NIR bands (B2, B3, B4, B8)
* `NDVI` = (B8 - B4) / (B8 + B4)
* `NDWI` = (B3 - B8) / (B3 + B8)

*(Note: Sentinel-2 array indices: B2=1, B3=2, B4=3, B8=7)*


In [ ]:
# TODO: Calculate Brightness, NDVI, and NDWI using the specified bands.
# TODO: Implement Min-Max scaling to normalize your 3 handcrafted indices AND the first 3 Principal Components to a [0, 1] range.



### 5. Evaluating the Math (L1 Loss)
Calculate the Mean Absolute Error (L1 Loss) to see how closely the abstract Principal Components match the physical equations.
* Compare PC1 vs Brightness
* Compare PC2 vs NDVI
* Compare PC3 vs NDWI

**Advisory:** The mathematical direction (sign) of a Principal Component eigenvector is arbitrary. Be mindful of inversely related data before calculating your error.


In [ ]:
def calc_l1_loss(true_arr, pred_arr):
    return np.mean(np.abs(true_arr - pred_arr))

# TODO: Calculate the L1 loss between the physical indices and their corresponding PCs. 
# Store the results in 'loss_pc1_bright', 'loss_pc2_ndvi', and 'loss_pc3_ndwi'.

loss_pc1_bright = 0.0 
loss_pc2_ndvi = 0.0 
loss_pc3_ndwi = 0.0 

print(f"L1 Loss (PC1 vs Brightness): {loss_pc1_bright:.2f}")
print(f"L1 Loss (PC2 vs NDVI): {loss_pc2_ndvi:.2f}")
print(f"L1 Loss (PC3 vs NDWI): {loss_pc3_ndwi:.2f}")


### 6. Analysis Questions

**Question 1:** Based on your Cumulative Variance calculations, what is the minimum number of Principal Components required to retain $\ge 95\%$ of the total variance in this dataset? 
*(You will add your integer answer to the final submission dictionary under the key `ans1`)*

**Question 2:** Analyze your L1 Loss results. Which PCs aligned well with known physical features, and which did not? Based on the properties of this dataset (a Level-1C image over an urban area), formulate a hypothesis explaining the physical meaning of the third principal component (PC3) and what PCA is isolating in this lower-variance component.
*(You will add your written analysis to the final submission dictionary under the key `ans2`)*


### 7. Submission Cell
Run this cell to generate your final submission payload.


In [ ]:
import json

submission_data = {
    "ids": [], #"your_student_id_here",
    "max_corr": round(float(max_corr), 2),
    "min_corr": round(float(min_corr), 2),
    "var_pc1": round(float(var_pc1), 2),
    "var_pc2": round(float(var_pc2), 2),
    "var_pc3": round(float(var_pc3), 2),
    "loss_pc1_bright": round(float(loss_pc1_bright), 2),
    "loss_pc2_ndvi": round(float(loss_pc2_ndvi), 2),
    "loss_pc3_ndwi": round(float(loss_pc3_ndwi), 2),
    "notes": [], # For any additional comments or observations you have about the results can go here
    "msgs": [] # For any messages you want to convey to the TAs  
}

print("AUTOGRADER_OUTPUT_START")
print(json.dumps(submission_data, indent=4))
print("AUTOGRADER_OUTPUT_END")


### 8. Bonus Challenge: Ground Truth Comparison (Level-2A)
You have just analyzed `hazy_cufe.tiff`, which is a Level-1C (Top of Atmosphere) image containing 13 bands, including the B10 Cirrus band which is highly sensitive to atmospheric noise and haze.

Provided in your data folder is another image: `gt_cufe.tiff`. This is a Level-2A (Surface Reflectance) "Ground Truth" image that has been atmospherically corrected. 
* **Note:** The Level-2A image only has **12 bands** because the B10 Cirrus band is removed after atmospheric correction.

**The Challenge:**
1. Change the `image_path` in Cell 3 to `"gt_cufe.tiff"` and rerun the entire notebook. 
2. *Advisory:* Double-check your array indices for calculating Brightness, NDVI, and NDWI. Since the Level-2A image dropped a band, did the column indices for B2, B3, B4, and B8 shift, or did the dropped band come later in the sequence?
3. **Analyze the differences:** How does the removal of atmospheric haze and the B10 band affect the Variance Explained by the Principal Components? Specifically, look at PC3. What does PC3 represent now that the atmospheric noise is gone? Does its L1 loss against physical indices improve?

*If you complete this challenge, add your comparative analysis as a string inside the `"notes"` list in your final submission dictionary!*
